In [1]:
import json
import pandas as pd

In [2]:
batches_direct_spelling = ['10003120', '10003940', '10004561', '10004621', '10004652', '10004821', '10005454', '10007783', '10013259'] + \
    ['10014762', '10018083', '10018098', '10018103', '10018126', '10018175', '10022891']

In [3]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_flipside_chosen_epoch3.json', 'r') as f:
    eval_dict = json.load(f)

In [19]:
#evaluate pipeline after spell correct and instance choice
total_num_of_ind_pages = 0
total_num_of_hits = 0
total_num_of_amb = 0
total_num_of_false_pos = 0
no_of_plus = 0

with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_epoch3_flipside_choice_eval.txt', 'w') as f, open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_epoch3_flipside_choice_fp.txt', 'w') as f2:
    f.write('BATCH' + '\t' + 'HIT_RATE' + '\t' + 'AMB_RATE' + '\t' + 'FP_RATE' + '\n')

    for batch in eval_dict.keys():

        batch_num_of_ind_pages = 0
        batch_num_of_hits = 0
        batch_num_of_amb = 0
        batch_num_of_false_pos = 0

        i = 0

        for page in eval_dict[batch].keys():

            i += 1

            if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:

                total_num_of_ind_pages += 1
                batch_num_of_ind_pages += 1

                gts_pred = [x[0] for x in eval_dict[batch][page][0]['gts']]

                for gt in gts_pred:
                    if '+' in gt:
                        no_of_plus += 1
                        break

                for inst in eval_dict[batch][page]:
                    gts_pred = [x[0] for x in inst['gts']]
                    
                    gts_fnr = [x[1] for x in inst['gts']]

                    if 'chosen' in inst.keys() and inst['chosen'] == True:

                        false_pos = True
                        
                        if len(inst['pred_links']) > 1:
                            total_num_of_amb +=1
                            batch_num_of_amb +=1
                        elif len(inst['sc_links']) > 1:
                            total_num_of_amb +=1
                            batch_num_of_amb +=1

                        

                        for link in inst['pred_links']:
                            if link[1] in gts_fnr:
                                false_pos = False
                                
                                #här ska villkor in
                                if float(inst['pred_conf']) > 0.98:
                                    total_num_of_hits += 1
                                    batch_num_of_hits += 1
                                
                                break
                        for link in inst['sc_links']:
                            
                            if link[1] in gts_fnr:
                                false_pos = False
                                
                                if inst['sc_pred'][1][2] >= 1 and inst['sc_pred'][1][1] > 0.7 and inst['sc_pred'][1][0] > 0.7 and float(inst['pred_conf']) > 0.95:
                                    total_num_of_hits += 1
                                    batch_num_of_hits += 1
                                
                                break


                        if false_pos and ((float(inst['pred_conf']) > 0.98 and len(inst['pred_links']) > 0) or (len(inst['sc_links']) > 0 and inst['sc_pred'][1][2] >= 1 and inst['sc_pred'][1][1] > 0.7 and inst['sc_pred'][1][0] > 0.7 and float(inst['pred_conf']) > 0.95)):
                            if i % 2 == 1:
                                try:
                                    f2.write(page + ': ' + inst['pred_links'][0][0] + ' :direct' + ': ' + str(gts_pred) + inst['pred_conf'] +'\n')
                                except:
                                    pass
                                try:
                                    f2.write(page + ': ' + inst['sc_links'][0][0] + ': sc' + ': ' + str(gts_pred) + inst['pred_conf'] + '\n')
                                except:
                                    pass
                            
                            total_num_of_false_pos += 1
                            batch_num_of_false_pos += 1

        hit_rate = str(batch_num_of_hits/(batch_num_of_ind_pages/2))
        amb_rate = str(batch_num_of_amb/(batch_num_of_ind_pages/2))
        fp_rate = str(batch_num_of_false_pos/(batch_num_of_ind_pages/2))

        f.write(batch + '\t' + hit_rate + '\t' + amb_rate + '\t' + fp_rate + '\n')
        f2.write('\n')
        

print((total_num_of_hits)/(total_num_of_ind_pages/2))
print(total_num_of_amb/(total_num_of_ind_pages/2))
print((total_num_of_false_pos)/(total_num_of_ind_pages/2))



0.8706564869726887
0.03084894860516961
0.014202222432625517
